# TP3 — Évaluation : mesurer pour choisir

Depuis ce matin on a fait des choix : découpage, modèle d'embeddings, `k`, recherche sémantique ou hybride, prompt,
modèle de génération… et on les a jugés sur trois questions. Ce n'est pas une mesure.

Pour comparer objectivement deux variantes, il faut un **jeu de test** — des questions dont on connaît la réponse — et
des **métriques**. On évalue séparément les deux moitiés du RAG :

- le **retrieval** : le bon passage est-il retrouvé ? (rapide, gratuit, objectif) ;
- la **génération** : la réponse est-elle correcte ? (plus difficile : c'est du texte libre).

Plan :
1. Un jeu de questions
2. Évaluer le retrieval : hit@k et MRR
3. Comparer des variantes
4. Évaluer la génération avec un LLM juge
5. Exercices

In [ ]:
import json
import re

import numpy as np
import matplotlib.pyplot as plt
from dotenv import load_dotenv
from litellm import completion

import settings
from rag import load_index, load_pdfs, chunk_pages, embed, search, search_bm25, answer

load_dotenv()
chunks, embeddings = load_index()

## 1. Un jeu de questions

Une question de test = `{"question", "source", "pages"}` : la réponse se trouve à l'une de ces pages de ce document
(une même information figure souvent à plusieurs endroits : résumé, tableau, texte).
On repère la réponse par sa **page** et non par un numéro de chunk : le jeu reste valable si on change le découpage.

Le fichier `eval/questions.json` contient une vingtaine de questions écrites à la main et vérifiées.

In [ ]:
with open("eval/questions.json", encoding="utf-8") as f:
    jeu_test = json.load(f)

print(len(jeu_test), "questions")
for item in jeu_test[:5]:
    print(f"- {item['question']}  →  {item['source']} p.{item['pages']}")

Écrire des questions à la main est long. On peut en faire **générer** par le LLM : on tire des chunks au hasard et on lui
demande une question à laquelle le chunk répond. La vérité terrain est la page du chunk.

*Cellule optionnelle (~20 appels à l'API).* Regardez les questions produites : certaines seront mauvaises (chunk de tableau
illisible, question ambiguë). La qualité du jeu de test conditionne tout ce qui suit — en pratique on relit et on trie.

In [ ]:
import random

PROMPT_QUESTION = """Voici un extrait d'une datasheet de composant électronique :

---
{texte}
---

Écris UNE question précise, en français, à laquelle cet extrait permet de répondre (une valeur, une caractéristique, un comportement).
La question doit se comprendre seule : nomme le composant. Réponds uniquement par la question."""

random.seed(0)
echantillon = [c for c in random.sample(chunks, 40) if len(c["text"].split()) > 100][:20]

questions_synthetiques = []
for c in echantillon:
    reponse = completion(
        model=settings.GENERATION_MODEL,
        messages=[{"role": "user", "content": PROMPT_QUESTION.format(texte=c["text"])}],
        temperature=0, max_tokens=100,
    )
    question = reponse.choices[0].message.content.strip()
    questions_synthetiques.append({"question": question, "source": c["source"], "pages": [c["page"]]})
    print(f"- {question}  →  {c['source']} p.{c['page']}")

# Pour les ajouter au jeu de test après relecture :
# jeu_test += questions_synthetiques

## 2. Évaluer le retrieval : hit@k et MRR

Pour chaque question on lance la recherche et on regarde **à quel rang** apparaît la bonne page.

- **hit@k** : proportion de questions pour lesquelles la bonne page est dans les k premiers résultats.
  hit@5 = 0,8 signifie que dans 80 % des cas, le LLM aura la bonne page sous les yeux avec `k = 5`.
- **MRR** (*Mean Reciprocal Rank*) : moyenne de 1/rang de la première bonne page (0 si absente).
  Plus la bonne page est haut, mieux c'est.

In [ ]:
def evaluer_retrieval(fonction_recherche, jeu_test, k_max=10):
    # fonction_recherche(question, k) -> liste de chunks ; renvoie (hit@1..k_max, MRR)
    hits = np.zeros(k_max)
    rangs_inverses = []
    for item in jeu_test:
        resultats = fonction_recherche(item["question"], k_max)
        rangs = [rang for rang, r in enumerate(resultats, start=1)
                 if r["source"] == item["source"] and r["page"] in item["pages"]]
        premier = rangs[0] if rangs else None
        rangs_inverses.append(1 / premier if premier else 0)
        for k in range(1, k_max + 1):
            hits[k - 1] += premier is not None and premier <= k
    return hits / len(jeu_test), float(np.mean(rangs_inverses))


semantique = lambda q, k: search(q, chunks, embeddings, k)
mots_cles = lambda q, k: search_bm25(q, chunks, k)

resultats = {
    "sémantique": evaluer_retrieval(semantique, jeu_test),
    "BM25": evaluer_retrieval(mots_cles, jeu_test),
}
for nom, (hits, mrr) in resultats.items():
    print(f"{nom:<12} hit@1={hits[0]:.2f}  hit@5={hits[4]:.2f}  hit@10={hits[9]:.2f}  MRR={mrr:.2f}")

In [ ]:
for nom, (hits, mrr) in resultats.items():
    plt.plot(range(1, 11), hits, marker="o", label=f"{nom} (MRR={mrr:.2f})")
plt.xlabel("k"); plt.ylabel("hit@k"); plt.ylim(0, 1.05); plt.grid(alpha=0.3); plt.legend()
plt.title("La bonne page est-elle dans les k premiers résultats ?")
plt.show()

**À observer**

- La courbe monte avec k : plus on donne de chunks au LLM, plus la bonne page a des chances d'y être — au prix de tokens.
- Le point où la courbe plafonne aide à choisir `settings.TOP_K`.
- Comparez sémantique et BM25 : lequel gagne sur ce jeu ? Cela dépend des questions (identifiants exacts ou paraphrases),
  de la langue et du découpage — d'où l'intérêt d'un jeu de test **représentatif des vraies questions des utilisateurs**.

Si vous avez écrit `search_hybrid` au TP1, ajoutez-la à la comparaison.

In [ ]:
# from tp1 : collez ici votre search_hybrid, puis :
# resultats["hybride"] = evaluer_retrieval(lambda q, k: search_hybrid(q, chunks, embeddings, k), jeu_test)

## 3. Comparer des variantes de découpage

Même méthode pour le découpage : on reconstruit un index avec d'autres paramètres et on mesure.
Ici des chunks deux fois plus grands (300 mots au lieu de 150). Le calcul des embeddings prend 1 à 2 minutes.

In [ ]:
pages = load_pdfs()
chunks_300 = chunk_pages(pages, chunk_size=300, overlap=50)
embeddings_300 = embed([c["text"] for c in chunks_300])
print(len(chunks_300), "chunks de 300 mots (contre", len(chunks), "de 150)")

resultats["sémantique, chunks 300"] = evaluer_retrieval(lambda q, k: search(q, chunks_300, embeddings_300, k), jeu_test)

for nom, (hits, mrr) in resultats.items():
    print(f"{nom:<24} hit@1={hits[0]:.2f}  hit@5={hits[4]:.2f}  hit@10={hits[9]:.2f}  MRR={mrr:.2f}")

**À observer**

Des chunks plus grands contiennent plus souvent la réponse (moins de coupures) mais sont moins précis et coûtent plus
de tokens par chunk. Il n'y a pas de bonne taille universelle : elle dépend des documents et des questions —
**et on la choisit en mesurant**, pas au feeling.

## 4. Évaluer la génération avec un LLM juge

La réponse du RAG est un texte libre : pas de métrique simple. La pratique courante est de demander à un LLM de
**juger** : on lui donne la question, le passage source (la vérité) et la réponse du RAG, il note de 0 à 10 et se justifie.

Limites à garder en tête : le juge peut se tromper, il a des biais (il aime les réponses longues), et il coûte des appels.
On vérifie toujours un échantillon à la main. Ici on utilise le gros modèle comme juge et on note 8 questions.

In [ ]:
PROMPT_JUGE = """Tu évalues la réponse d'un assistant à une question technique.

Question : {question}

Passage de référence (contient la bonne réponse) :
---
{reference}
---

Réponse de l'assistant :
---
{reponse}
---

Note la réponse de 0 (fausse ou hors sujet) à 10 (exacte et complète), en te basant uniquement sur le passage de référence.
Réponds au format JSON : {{"note": <entier>, "justification": "<une phrase>"}}"""

pages_par_cle = {(p["source"], p["page"]): p["text"] for p in load_pdfs()}

notes = []
for item in jeu_test[:8]:
    reponse_rag, _ = answer(item["question"], chunks, embeddings)
    prompt = PROMPT_JUGE.format(question=item["question"],
                                reference=pages_par_cle[(item["source"], item["pages"][0])][:3000],
                                reponse=reponse_rag)
    reponse_juge = completion(model=settings.GENERATION_MODEL_LARGE, messages=[{"role": "user", "content": prompt}],
                              temperature=0, max_tokens=150)
    verdict = json.loads(re.search(r"\{.*\}", reponse_juge.choices[0].message.content, re.S).group())  # extrait le JSON
    notes.append(verdict["note"])
    print(f"{verdict['note']:>2}/10  {item['question']}\n       {verdict['justification']}\n")

print("note moyenne :", np.mean(notes))

**À observer**

- Lisez les justifications : êtes-vous d'accord avec le juge ? Sur quelles questions le RAG échoue-t-il, et pourquoi
  (mauvais retrieval ? mauvaise lecture d'un tableau ? réponse trop prudente ?) ?
- Cette note moyenne devient votre **indicateur** : changez le prompt, `k`, le modèle… et re-mesurez.

## 5. Exercices

**Exercice 1 — mesurez vos améliorations du TP1.** Découpage amélioré, recherche hybride, autre modèle d'embeddings :
calculez hit@5 et MRR pour chaque variante. Quelle combinaison gagne ? (Comparez vos résultats entre groupes.)

**Exercice 2 — questions pièges.** Ajoutez au jeu de test des questions dont la réponse **n'est pas** dans le corpus
(`"source": null`). Mesurez la proportion de fois où le RAG dit correctement qu'il ne sait pas.

**Exercice 3 — un juge fiable ?** Faites juger deux fois la même réponse, ou par le petit modèle : les notes sont-elles
stables ? Donnez au juge un mauvais passage de référence : que se passe-t-il ?

## Bilan

```
jeu de test (question, page)  ──►  retrieval : hit@k, MRR      (gratuit, rapide : à lancer à chaque changement)
                              ──►  génération : LLM juge       (coûteux : sur un échantillon)
```

Un RAG n'est jamais "fini" : chaque réglage se décide en mesurant sur un jeu de test représentatif.
C'est ce qui distingue une démo qui marche sur trois questions d'un système utilisable.